# Assistente de Reclamações com IA — versão 4 e comparação de modelos

Continuação do notebook `01_classificacao_reclamacoes.ipynb`. Aqui testamos duas melhorias, **medindo cada uma separadamente**:

1. **Prompt versão 4:** baseado em um **guia de classificação** escrito a partir da análise humana dos erros (sinais de tempo no texto, critério de maior prioridade, campo "cliente procurou a loja"), com exemplos resolvidos e uma justificativa curta antes da resposta.
2. **Modelo melhor:** `gpt-4o-mini` × `gpt-5.4-mini`, comparando acerto **e custo**.

A avaliação usa um **conjunto de teste com 30 reclamações novas**, que não foram usadas em nenhum ajuste anterior.

## 1. Configuração

Envie para o Colab (ícone de pasta 📁):
- `reclamacoes_classificadas_final.csv` (resultado da versão 3, gerado no notebook 01)

A chave da API continua nos *Secrets* (🔑), com o nome `OPENAI_API_KEY`.

In [1]:
!pip install -q openai

In [3]:
import json, os, time, unicodedata
import pandas as pd
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass

client = OpenAI()

# Preço por 1 milhão de tokens (US$), conforme a página de modelos da OpenAI em nov/2026
PRECOS = {
    "gpt-4o-mini":  {"entrada": 0.15, "saida": 0.60},
    "gpt-5.4-mini": {"entrada": 0.75, "saida": 4.50},
}

base = pd.read_csv("reclamacoes_classificadas_final.csv", encoding="utf-8-sig")
print(f"{len(base)} reclamações carregadas")

200 reclamações carregadas


## 2. Conjunto de teste

As linhas 0 a 29 da amostra foram o **gabarito** (usado para ajustar os prompts) e as linhas 30 a 49, a **validação**. O teste usa as linhas **50 a 79**, que ninguém viu ainda.

1. Rode a célula abaixo e baixe `teste_para_preencher.csv`.
2. Preencha **seguindo o guia de classificação** (impresso abaixo) e **sem olhar a IA**: número do tema (1 a 8) e urgência (`alta`, `media` ou `baixa`).
3. Envie de volta como `teste_preenchido.csv`.

In [4]:
GUIA = """
TEMAS (escolha um)
1. Atraso na entrega: o produto CHEGOU, mas depois do prazo prometido.
2. Produto não entregue: o cliente diz que AINDA NÃO RECEBEU o pedido (inclui "aguardando", "esperando", "não chegou", "consta entregue mas não recebi").
3. Produto errado ou incompleto: chegou item diferente do comprado ou faltou parte do pedido (ex.: comprou 2 e recebeu 1).
4. Produto com defeito ou avariado: chegou quebrado, danificado ou não funciona.
5. Qualidade abaixo do esperado: funciona, mas é pior do que o anunciado (material, tamanho, acabamento).
6. Cobrança, pagamento ou reembolso: valor, estorno, nota fiscal ou devolução do dinheiro.
7. Atendimento ou falta de resposta: SÓ quando a queixa é apenas sobre o atendimento, sem outro problema por trás.
8. Outros: nenhum dos anteriores, incluindo comentário elogioso ou sem problema, ter que retirar o produto nos Correios, problema de frete sem atraso e crítica genérica à loja.

REGRAS DE TEMA
- Se a reclamação tiver mais de um problema, o tema é o de MAIOR PRIORIDADE (maior urgência); o outro vai como tema secundário.
- Se o cliente já procurou a loja e não teve resposta, o tema continua sendo o problema de origem (ex.: não entregue) e isso é registrado no campo "cliente_procurou_loja".

URGÊNCIA
Alta:
- o cliente pagou e está sem o que comprou com atraso confirmado: "já passou do prazo", "até hoje não recebi", cita datas ou muito tempo de espera;
- produto errado ou faltando parte do pedido;
- cobrança indevida;
- o cliente já procurou a loja e não teve solução, cita reincidência, ameaça Procon/processo ou diz que não compra mais.
Média:
- espera sem prazo confirmado, mas com sinal de demora: "AINDA estou esperando", "CONTINUO esperando" (vale verificar em momento oportuno);
- produto com defeito ou avariado;
- outro problema que ainda precisa de solução da loja.
Baixa:
- espera simples, sem sinal de demora: "estou aguardando o produto" (pode estar dentro do prazo);
- problema já resolvido (ex.: atrasou, mas chegou);
- insatisfação com qualidade sem pedido de solução, elogio ou crítica genérica.
"""

teste = base.loc[50:79, ["review_id", "texto"]].copy()
teste["tema_humano"] = ""
teste["urgencia_humana"] = ""
teste.to_csv("teste_para_preencher.csv", index=False, sep=";", encoding="utf-8-sig")

print(f"{len(teste)} reclamações para preencher.\n")
print(GUIA)

30 reclamações para preencher.


TEMAS (escolha um)
1. Atraso na entrega: o produto CHEGOU, mas depois do prazo prometido.
2. Produto não entregue: o cliente diz que AINDA NÃO RECEBEU o pedido (inclui "aguardando", "esperando", "não chegou", "consta entregue mas não recebi").
3. Produto errado ou incompleto: chegou item diferente do comprado ou faltou parte do pedido (ex.: comprou 2 e recebeu 1).
4. Produto com defeito ou avariado: chegou quebrado, danificado ou não funciona.
5. Qualidade abaixo do esperado: funciona, mas é pior do que o anunciado (material, tamanho, acabamento).
6. Cobrança, pagamento ou reembolso: valor, estorno, nota fiscal ou devolução do dinheiro.
7. Atendimento ou falta de resposta: SÓ quando a queixa é apenas sobre o atendimento, sem outro problema por trás.
8. Outros: nenhum dos anteriores, incluindo comentário elogioso ou sem problema, ter que retirar o produto nos Correios, problema de frete sem atraso e crítica genérica à loja.

REGRAS DE TEMA
- Se a reclama

## 3. Prompt versão 4

O prompt usa **o mesmo guia** que a pessoa usou para classificar, mais seis exemplos resolvidos tirados do gabarito e da validação (nunca do teste). A IA escreve uma justificativa curta **antes** de escolher o tema, o que costuma reduzir erros.

In [5]:
TEMAS = [
    "Atraso na entrega",
    "Produto não entregue",
    "Produto errado ou incompleto",
    "Produto com defeito ou avariado",
    "Qualidade abaixo do esperado",
    "Cobrança, pagamento ou reembolso",
    "Atendimento ou falta de resposta",
    "Outros",
]

EXEMPLOS = """
Reclamação: "estou aguardando o produto chegar"
→ tema: Produto não entregue · urgencia: baixa · cliente_procurou_loja: nao
  (espera simples, pode estar dentro do prazo)

Reclamação: "ainda estou esperando meu pedido."
→ tema: Produto não entregue · urgencia: media · cliente_procurou_loja: nao
  ("ainda" indica demora, mas sem atraso confirmado)

Reclamação: "até hoje não recebi o produto, já passou do prazo de entrega e nada ainda"
→ tema: Produto não entregue · urgencia: alta · cliente_procurou_loja: nao
  (atraso confirmado)

Reclamação: "o produto atrasou demais na entrega e além disso a entrega não foi completa, pois faltou um cartucho de tinta colorido"
→ tema: Produto errado ou incompleto · tema_secundario: Atraso na entrega · urgencia: alta · cliente_procurou_loja: nao
  (dois problemas: o tema é o de maior prioridade)

Reclamação: "luminária faltando, comprei 2 luminárias e só mandaram 1, mandei um email falando do problema e até agora ninguém respondeu"
→ tema: Produto errado ou incompleto · urgencia: alta · cliente_procurou_loja: sim
  (o tema é o problema de origem; a falta de resposta vai no campo próprio)

Reclamação: "Eu gostei muito dos meus produto"
→ tema: Outros · urgencia: baixa · cliente_procurou_loja: nao
  (comentário sem problema)
"""

INSTRUCOES_V4 = f"""
Você é um analista de atendimento ao cliente de um e-commerce brasileiro.
Classifique a reclamação seguindo EXATAMENTE o guia abaixo.

GUIA DE CLASSIFICAÇÃO
{GUIA}

EXEMPLOS RESOLVIDOS
{EXEMPLOS}

Devolva APENAS um JSON com os campos, nesta ordem:
- justificativa: uma frase curta explicando o tema e a urgência, citando o trecho decisivo
- tema: um destes valores exatos: {TEMAS}
- tema_secundario: outro valor da mesma lista, ou null
- urgencia: "alta", "media" ou "baixa"
- cliente_procurou_loja: "sim" ou "nao"
- sentimento: "muito negativo", "negativo" ou "neutro"
- risco_perder_cliente: "alto", "medio" ou "baixo", avaliado SEPARADAMENTE da urgência
    alto  = diz que não compra mais, pede cancelamento, ameaça Procon/processo,
            cita reincidência ou usa tom de revolta
    medio = está insatisfeito, mas ainda espera uma solução da loja
    baixo = crítica pontual, em tom moderado
- resumo: uma frase curta com o problema principal
- acao_sugerida: o próximo passo concreto para o time de atendimento
- resposta_sugerida: resposta educada e empática ao cliente, em português,
  com no máximo 70 palavras. Não prometa prazos ou valores específicos
  e não invente informações que não estão na reclamação.
"""

## 4. Classificação com os dois modelos

Os modelos da família GPT-5 raciocinam antes de responder e não aceitam `temperature`; no lugar, usam `reasoning_effort`. A função trata os dois casos e **registra os tokens gastos**, para calcular o custo de cada modelo.

In [6]:
def classificar(texto: str, modelo: str, instrucoes: str):
    """Classifica uma reclamação e devolve (resultado, tokens de entrada, tokens de saída)."""
    parametros = dict(
        model=modelo,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": instrucoes},
                  {"role": "user", "content": texto}],
    )
    if modelo.startswith("gpt-5"):
        parametros["reasoning_effort"] = "low"    # modelos com raciocínio
    else:
        parametros["temperature"] = 0             # modelos sem raciocínio

    for tentativa in range(3):
        try:
            resp = client.chat.completions.create(**parametros)
            uso = resp.usage
            return json.loads(resp.choices[0].message.content), uso.prompt_tokens, uso.completion_tokens
        except Exception as erro:
            print(f"Tentativa {tentativa + 1} falhou: {erro}")
            time.sleep(2 ** tentativa)
    return {"tema": "ERRO"}, 0, 0


def classificar_lote(textos, modelo, instrucoes):
    resultados, entrada, saida = [], 0, 0
    inicio = time.time()
    for i, texto in enumerate(textos, start=1):
        r, e, s = classificar(texto, modelo, instrucoes)
        resultados.append(r); entrada += e; saida += s
        if i % 10 == 0:
            print(f"  {modelo}: {i} de {len(textos)}")
    preco = PRECOS[modelo]
    custo = entrada / 1e6 * preco["entrada"] + saida / 1e6 * preco["saida"]
    info = {"custo_por_1000": custo / len(textos) * 1000,
            "segundos_por_reclamacao": (time.time() - inicio) / len(textos)}
    return pd.DataFrame(resultados), info

# Teste rápido com uma reclamação em cada modelo
for m in PRECOS:
    r, e, s = classificar("ainda estou esperando meu pedido.", m, INSTRUCOES_V4)
    print(m, "→", r.get("tema"), "·", r.get("urgencia"), "·", r.get("justificativa"))

gpt-4o-mini → Produto não entregue · media · O cliente menciona que 'ainda' está esperando, indicando uma espera sem prazo confirmado, mas com sinal de demora.
gpt-5.4-mini → Produto não entregue · media · O cliente diz "ainda estou esperando meu pedido", indicando produto não entregue com sinal de demora, mas sem prazo confirmado.


## 5. Avaliação no conjunto de teste

In [7]:
def normalizar(valor):
    texto = unicodedata.normalize("NFKD", str(valor)).encode("ascii", "ignore").decode()
    return texto.strip().lower()

def ler_gabarito(arquivo: str) -> pd.DataFrame:
    g = pd.read_csv(arquivo, sep=None, engine="python", encoding="utf-8-sig", dtype=str)
    g = g.dropna(subset=["tema_humano", "urgencia_humana"]).reset_index(drop=True)
    g["tema_humano"] = g["tema_humano"].str.strip().astype(int).map(lambda n: TEMAS[n - 1])
    g["urgencia_humana"] = g["urgencia_humana"].map(normalizar)
    invalidos = g[~g["urgencia_humana"].isin(["alta", "media", "baixa"])]
    if len(invalidos):
        print("⚠️ Urgência digitada errado (corrija no CSV e rode de novo):")
        print(invalidos[["review_id", "urgencia_humana"]], "\n")
    return g

def concordancia(gabarito, previsto):
    b = gabarito[["review_id", "texto", "tema_humano", "urgencia_humana"]].merge(
        previsto[["review_id", "tema", "urgencia"]], on="review_id")
    b["urgencia"] = b["urgencia"].map(normalizar)
    acerto = {"tema": (b["tema"] == b["tema_humano"]).mean(),
              "urgencia": (b["urgencia"] == b["urgencia_humana"]).mean()}
    erros = b[(b["tema"] != b["tema_humano"]) | (b["urgencia"] != b["urgencia_humana"])]
    return acerto, erros

teste_preenchido = ler_gabarito("teste_preenchido.csv")
ids = teste_preenchido[["review_id"]]

resultados, erros = {}, {}

# Versão 3 + gpt-4o-mini: já classificada no notebook 01 (sem custo)
acerto, erros["v3 · gpt-4o-mini"] = concordancia(teste_preenchido, base)
resultados["v3 · gpt-4o-mini"] = {**acerto, "custo_por_1000": None, "segundos_por_reclamacao": None}

# Versão 4 nos dois modelos
for modelo in PRECOS:
    df, info = classificar_lote(teste_preenchido["texto"], modelo, INSTRUCOES_V4)
    previsto = pd.concat([ids, df], axis=1)
    previsto.to_csv(f"teste_v4_{modelo}.csv", index=False, encoding="utf-8-sig")
    nome = f"v4 · {modelo}"
    acerto, erros[nome] = concordancia(teste_preenchido, previsto)
    resultados[nome] = {**acerto, **info}

comparacao = pd.DataFrame(resultados).T
comparacao.to_csv("comparacao_modelos.csv", encoding="utf-8-sig")
comparacao.style.format({"tema": "{:.0%}", "urgencia": "{:.0%}",
                         "custo_por_1000": "US$ {:.2f}", "segundos_por_reclamacao": "{:.1f} s"}, na_rep="—")

  gpt-4o-mini: 10 de 30
  gpt-4o-mini: 20 de 30
  gpt-4o-mini: 30 de 30
  gpt-5.4-mini: 10 de 30
  gpt-5.4-mini: 20 de 30
  gpt-5.4-mini: 30 de 30


,tema,urgencia,custo_por_1000,segundos_por_reclamacao
v3 · gpt-4o-mini,67%,73%,—,—
v4 · gpt-4o-mini,70%,80%,US$ 0.30,2.3 s
v4 · gpt-5.4-mini,70%,67%,US$ 2.41,2.3 s


**Onde cada versão ainda erra:** troque o nome entre colchetes para ver os erros de cada combinação.

In [8]:
pd.set_option("display.max_colwidth", 120)
erros["v4 · gpt-5.4-mini"]

,review_id,texto,tema_humano,urgencia_humana,tema,urgencia
0,6e755cbed347777aa1d6e8bd73424869,Demora muito pra chegar,Atraso na entrega,baixa,Produto não entregue,media
1,89648dafebe6ed56862072cae885e238,A almofada é meio murcha e o enchimento é com uns pedaços de espuma soltos dentro dela. Não dá firmeza para o bebê.,Outros,baixa,Qualidade abaixo do esperado,baixa
2,d751bab5e874f66a91b0d3e8bd9b6af0,Quero meu produto Nota não tenho pra da quero meu produto,Produto não entregue,alta,Produto não entregue,baixa
3,057891cc832006e28868b88fa1236fbc,mas poderia ter sido mais rapido,Atraso na entrega,baixa,Outros,baixa
5,a89701ddb37339ce9758120cfb51d67f,Produto não corresponde a imagem de venda,Produto errado ou incompleto,alta,Qualidade abaixo do esperado,baixa
6,1f57ecb3f93be658c338ed216bab4096,"não recomendo pois não entregou no prazo e não consigo falar com eles, é muito ruim a targaryen",Atendimento ou falta de resposta,alta,Atraso na entrega,alta
12,e38e4439b18699515ef50e3554711c88,Nao recebi nem o produto nem o reembolso.,Produto não entregue,alta,"Cobrança, pagamento ou reembolso",alta
15,83c4f54bd18decda21d563b5ddf5558a,Recebi o produto com defeito,Produto com defeito ou avariado,alta,Produto com defeito ou avariado,media
16,abf61b99bc7101baaae96052d4a64519,Produto não chegou e não obrive resposta de nenhuma das duas solicitações que enviei para site da lannister e nem da...,Atendimento ou falta de resposta,alta,Produto não entregue,media
20,74b1f697396823d14fdde162d7144cc0,"n foi entregue o produto que comprei, e inclusive ja estou pagando pelo mesmo que n foi entregue.",Produto não entregue,alta,Produto não entregue,media


## 6. Classificação final com a melhor combinação

Escolha a combinação pelo **equilíbrio entre acerto e custo**, não só pelo acerto. Ajuste `MODELO_FINAL` abaixo e rode para gerar a base do painel.

In [9]:
MODELO_FINAL = "gpt-4o-mini"   # troque para "gpt-5.4-mini" se o ganho compensar o custo

df, info = classificar_lote(base["texto"], MODELO_FINAL, INSTRUCOES_V4)
final_v4 = pd.concat([base[["review_id", "review_score", "review_creation_date", "texto"]], df], axis=1)
final_v4.to_csv("reclamacoes_classificadas_v4.csv", index=False, encoding="utf-8-sig")

print(final_v4["tema"].value_counts(), "\n")
print(final_v4["urgencia"].value_counts(), "\n")
print("Cliente já procurou a loja:")
print(final_v4["cliente_procurou_loja"].value_counts(), "\n")
pd.crosstab(final_v4["urgencia"], final_v4["risco_perder_cliente"])

  gpt-4o-mini: 10 de 200
  gpt-4o-mini: 20 de 200
  gpt-4o-mini: 30 de 200
  gpt-4o-mini: 40 de 200
  gpt-4o-mini: 50 de 200
  gpt-4o-mini: 60 de 200
  gpt-4o-mini: 70 de 200
  gpt-4o-mini: 80 de 200
  gpt-4o-mini: 90 de 200
  gpt-4o-mini: 100 de 200
  gpt-4o-mini: 110 de 200
  gpt-4o-mini: 120 de 200
  gpt-4o-mini: 130 de 200
  gpt-4o-mini: 140 de 200
  gpt-4o-mini: 150 de 200
  gpt-4o-mini: 160 de 200
  gpt-4o-mini: 170 de 200
  gpt-4o-mini: 180 de 200
  gpt-4o-mini: 190 de 200
  gpt-4o-mini: 200 de 200
tema
Produto não entregue                71
Produto errado ou incompleto        52
Qualidade abaixo do esperado        25
Produto com defeito ou avariado     21
Atraso na entrega                   13
Outros                              10
Atendimento ou falta de resposta     4
Cobrança, pagamento ou reembolso     4
Name: count, dtype: int64 

urgencia
alta     121
media     42
baixa     37
Name: count, dtype: int64 

Cliente já procurou a loja:
cliente_procurou_loja
nao    164
sim    

risco_perder_cliente,alto,baixo,medio
urgencia,,,
alta,47,0,74
baixa,1,34,2
media,1,0,41
